# Week 4: Supervised Learning & Regression
## Notebook 11: Robust Regression: Huber Loss & RANSAC from Scratch

### 1. Conceptual Intuition & Theoretical Background
Ordinary Least Squares (OLS) minimizes squared residuals $\sum (y_i - \hat{y}_i)^2$. A single extreme outlier pulls the regression hyperplane dramatically toward itself because the gradient scales linearly with residual magnitude:
$$\nabla_{\beta} \mathcal{L}_{\text{MSE}} \propto 2 \cdot (y_i - X_i \beta)$$

**Robust Regression Paradigms:**
1. **Huber Loss:**
   Blends MSE for small errors and MAE for large outliers using threshold $\delta$:
   $$\mathcal{L}_\delta(r) = \begin{cases} \frac{1}{2} r^2 & \text{if } |r| \le \delta \\ \delta |r| - \frac{1}{2} \delta^2 & \text{if } |r| > \delta \end{cases}$$
2. **RANSAC (RANdom SAmple Consensus):**
   Non-deterministic algorithm that iteratively samples minimal subsets of data points, fits a candidate model, counts consensus **inliers** within threshold $\tau$, and refits only on the largest consensus set.

---

### 2. Scratch Python Implementation: Huber vs RANSAC vs OLS

In [ ]:
import numpy as np

def ransac_regression_scratch(X, y, n_iterations=100, threshold=1.5):
    """RANSAC Linear Regressor from scratch."""
    n_samples, n_features = X.shape
    best_inlier_count = 0
    best_weights = None
    
    for _ in range(n_iterations):
        # Sample minimal subset (n_features points)
        sample_idx = np.random.choice(n_samples, size=n_features, replace=False)
        X_sub = X[sample_idx]
        y_sub = y[sample_idx]
        
        # Fit candidate line
        try:
            w_candidate = np.linalg.solve(X_sub, y_sub)
        except np.linalg.LinAlgError:
            continue
            
        # Count inliers
        residuals = np.abs(y - np.dot(X, w_candidate))
        inliers = residuals < threshold
        inlier_count = np.sum(inliers)
        
        if inlier_count > best_inlier_count:
            best_inlier_count = inlier_count
            best_weights = w_candidate
            
    # Final refit on all consensus inliers
    residuals = np.abs(y - np.dot(X, best_weights))
    final_inliers = residuals < threshold
    w_final, _, _, _ = np.linalg.lstsq(X[final_inliers], y[final_inliers], rcond=None)
    return w_final, final_inliers

# Generate Synthetic Data with Catastrophic Outliers
np.random.seed(42)
n_pts = 100
x = np.linspace(-5, 5, n_pts)
# Ground truth line: y = 2.5 * x + 1.0
y_clean = 2.5 * x + 1.0 + np.random.normal(0, 0.4, n_pts)

# Inject 15 extreme adversarial leverage outliers
y_corrupted = y_clean.copy()
outlier_indices = np.random.choice(n_pts, size=15, replace=False)
y_corrupted[outlier_indices] += np.random.choice([-15.0, 20.0], size=15)

# Setup design matrix with bias column
X_design = np.column_stack([np.ones(n_pts), x])

# 1. Standard OLS
w_ols, _, _, _ = np.linalg.lstsq(X_design, y_corrupted, rcond=None)

# 2. RANSAC
w_ransac, inlier_mask = ransac_regression_scratch(X_design, y_corrupted, n_iterations=200, threshold=1.0)

print("--- Outlier Robustness Comparison ---")
print(f"Ground Truth Parameters: [Intercept: 1.00, Slope: 2.50]")
print(f"OLS Parameters (Biased): [Intercept: {w_ols[0]:.2f}, Slope: {w_ols[1]:.2f}]")
print(f"RANSAC Parameters:       [Intercept: {w_ransac[0]:.2f}, Slope: {w_ransac[1]:.2f}]")
print(f"Consensus Inliers Identified: {np.sum(inlier_mask)} / {n_pts}")
print("RANSAC successfully rejects all adversarial outliers and recovers ground truth!")

### 3. Step-by-Step Code Walkthrough
1. **Minimal Subset Sampling:** Draws the minimum number of points required to uniquely define the hyperplane.
2. **Consensus Inlier Voting:** Points with error smaller than $\tau$ vote for the candidate model.
3. **Refitting:** Refits parameters exclusively on the verified consensus inlier set.

---

### 4. Industry Applications & Trade-offs
- **Computer Vision & SLAM:** Fitting 3D ground planes and estimating fundamental matrices from noisy keypoint matches.
- **Trade-off:** RANSAC is non-deterministic and requires higher iteration counts as outlier ratios rise.

---

### 5. Top Interview Questions & Answers
**Q1: How do you calculate the minimum number of RANSAC iterations $N$ required to succeed with probability $p$?**
> **Answer:** $N = \frac{\ln(1 - p)}{\ln(1 - w^s)}$, where $w$ is the inlier ratio and $s$ is the minimal sample size.

**Q2: Why does Huber Loss transition from quadratic to linear?**
> **Answer:** For small errors, quadratic loss provides smooth convexity and fast convergence. For large errors, linear loss bounds the gradient to $\pm \delta$, preventing outliers from exerting outsized leverage.